Lets go through the novel components of the Transformer architecture in more detail, starting with positional encodings...

**Positional Encodings:**
- A positional encoding is a dense vector that encodes the position of a token within a sentence: the ith positional encoding is added to the token embedding of the ith token in each sentence/ A simple way to implement this is to use an Embedding layer: just add embedding #0 to the representation of token #0 and embedding #1 to the representation of token #1. Alternatively, you could use an nn.Parameter to store the embedding matrix (initialized using random weights), then add its first L rows to the inputs where L is the max input sequence length. can also add ropout to reduce the risk of overfitting

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as f

remember:
- putting a plain tensor inside nn.Module implies that pytorch will ignore it during training
- wrapping it with nn.Parameter -> pytirch will register it as a model parameter, include it in model.parameters(), compute geadients for it, update it when u call optimizer.step()
- **Core idea:** nn.Parameter is the explicit signal for pytorch to be able to distinguish between learnable weights and fixed tensors/buffers


- Style used below is learned absolute positional embeddings - used in modern transformers such GPT and BERT style
- same crux: self-attention has no notion of order, if u permute tokens, attention gives the same result
- so we inject position information by adding a position-dependent vector to each token embedding.
- This class below learns those vectors directly

In [2]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1): # max_len: maximum seq length the model can handle, #embed_dim: model dimension, # dropout: regularization
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(max_length,embed_dim)*0.02)
        self.dropout = nn.Dropout(dropout)
    def forward(self, X):
        return self.dropout(X + self.pos_embed[:X.size(1)])

- we instantiate the maximum length the model will be able to handle as opposed to the length of the current input batch -  because parameters must exist before the forward pass, and their shape must be fixed. 
- 

Note:
- the inputs have a shape [batch size, seq length, embedding size], but we are adding positional encodings of shape [seq length, embedding size]. This works thanks to the broadcasting rules: the ith positional embedding is added to the ith token's representation of each sentence in the batch

Authors of the transformer paper also proposed using fixed positional encodings rather than trainable ones. Their approach used a pretty smart scheme based on the sine and cosine functions, but its not much used anymore, as it doesn't really perform any better than trainable positional embeddings (except perhaps on small transformers if you're lucky)

In [3]:
embed_dim = 512
max_length = 500
pos_embedding = PositionalEmbedding(max_length, embed_dim)
embeddings = torch.randn(256, 500, 512)
embeddings_with_pos = pos_embedding(embeddings)
embeddings_with_pos.shape

torch.Size([256, 500, 512])

In [4]:
z = nn.Parameter(torch.randn(200,512))

In [5]:
embeddings = torch.randn(256, 300, 512)
embeddings_with_pos = pos_embedding(embeddings)
embeddings_with_pos.shape

torch.Size([256, 300, 512])

Now lets look deeper into the heart of the transformer model: the multi-head attention layer.

**Multi-Head Attention:**
- The multi-head attention (MHA) layer is based on a scaled dot-product attention, a variant of the dot-product attention that scales down the similarity score by a constant factor. Attention(Q, K, V) 
- Possible to mask out some key/value pairs by adding a very large negative value to the correspoding similarity scores, just before computing softmax (in practice, we can add - torch.inf). The resulting weights will be equal to zero. This is useful to mask padding tokens as well as future tokens in the masked multi-head attention layer.
- PyTorch comes with the F.scaled_dot_product_attention() function - its inputs are just like Q, K, and , but these inputs can have extra dimensions 

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def split_heads(self, X):
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1,2) # output of shape (B, h, L, d)
    
    def forward(self, query, key, value):
        q = self.split_heads(self.q_proj(query)) # (B, h, Lq, d)
        k = self.split_heads(self.k_proj(key)) # (B, h, Lk, d)
        v = self.split_heads(self.v_proj(value)) # (B, h, Lv, d) with Lv = Lk
        scores = q@k.transpose(2,3)/self.d**0.5 # (B, h, Lq, Lk)
        weights = scores.softmax(dim = -1) # (B, h, Lq, Lk)
        Z = self.dropout(weights) @ v # (B, h, Lq, d)
        Z = Z.transpose(1,2) # (B, Lq, h, d)
        Z = Z.reshape(Z.size(0), Z.size(1), self.h*self.d) # (B, Lq, hxd)
        return (self.out_proj(Z), weights)

- the constructor stores the number of heads -> self.h and computes the number of dimensions per head - self.d (in order to maintain the required structure of the encoer in that it takes input embeddings of shape [B, seq_len, embed_dim] and produces output of the same shape but the embed_dim being more contextual, we split across the embed dimension by the total number of heads h that we have). Note: the embedding size must be divisible by the number of heads.
- We split along the embed dim across multiple heads: this way, each head can focus on specific xtics of the token.The first linear layers to the mha block let the model choose which xtics each head should focus on. 
- The split_heads() method is used in the forward method and it splits its input X along its last dimension (one split per head) - converting it from a 3D tensor of shape [B, L, hxd] to a 4D tensor of shape [B,L,h,d], where B is the batch size, L is the max length of the input sequences (specifically Lk for the key and Lq for the query), h is the number of heads, d is the number of dimensions per head (i.e. hxd = embedding size). The dimensions 1 and 2 are then swapped to get a tensor of shape [B, h, L, d]: since the matrix multiplication operator @ only works on the last 2 dimensions, it won't touch the first 2 dimensions B and h, so we will be able to use this operator to compute the scores across all instances in the batch and across all attention heads, all in one shot (q @ k.transpose(2,3)). 
- The forward() method starts by applying a linear transformation to the query, key, and value, anmd passes the result through the split_heads method. The next lines compute the raw dot product scores and then softmax (self attention across all heads) in one shot. plus we apply a bit of dropout on the weights. 
- next we ensure h and d are close to each other, then we reshape the tensor back to 3D: this will concatenate the outputs of all heads. we can then apply the output linear transformation and return the result along with the weights (incase they are needed later). 

We are now missing one important detail, masking. As discussed earlier, the decoder's masked self-attention layers must only consider previous tokens when trying to predict what the next token is (or else it would be cheating). Moreover, if the key contains padding tokens, we want to ignore them as well. so we need to update the forward method to support 2 additional arguments:
- attn_mask
  - A boolean mask of shape [Lq, Lk] that we will use to control which tokens in each query we should ignore.(True to ignore, false to attend)
- key_padding_mask
  - a boolean mask of shape [B, Lk] to locate the padding tokens in each key.

MHA with masking, could say masked multihead attention:

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def split_heads(self, X):
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1,2)

    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        scores = q@k.transpose(2,3)/self.d**0.5

        # masking support
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask, -torch.inf) # (B, h, Lq, Lk)
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2) # (B, 1, 1, Lk)
            scores = scores.masked_fill(mask, -torch.inf)
        
        weights = scores.softmax(dim=-1)
        Z = self.dropout(weights) @ v 
        Z = Z.transpose(1,2)
        Z = Z.reshape(Z.size(0), Z.size(1), self.h * self.d)
        return (self.out_proj(Z), weights)

above code replaces the scores we want to ignore with negative infinity, so the corresponding weights will be zero after the softmax operation. (if we tried to zero out these weights directly, the remaining weights would not add up to 1). Note that the masks are broadcastable automatically. attn_mask is broadcastable across the whole batch and all attention heads. and key padding mask is broadcastable across all heads and query tokens. 

**Building the rest of the transformer:**
- the rest of the transformer architecture is more straightforward. start with the encoder block

In [10]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        attn, _ = self.self_attn(src,src,src,attn_mask=src_mask,key_padding_mask=src_key_padding_mask)
        Z = self.norm1(src + self.dropout(attn))
        ff = self.dropout(\
                        self.linear2(\
                            self.dropout(\
                                self.linear1(Z).relu()
                                )))
        return self.norm2(Z + ff)

The feedforward block is composed of a first linear layer that expands the dimensionality to 2048 (by default), followed by a nonlinearity (relu in this case) - then a second linear layer that projects the data back to the original embedding size (also called the model dimension, d_model). This reverse bottleneck increases the expressive power of the nonlinearity, allowing the model learn much richer combination of features. 
- In the encoder, the src_mask argument is generally not used, since the encoder allows each token to attend all tokens even the ones located after it. However, the key_padding_mask is expected to be set and used appropriately

Decoder block:

In [11]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.multihead_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.dropout = nn.Dropout(dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
    
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        # masked self attention
        attn1, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask)
        Z = self.norm1(tgt + self.dropout(attn1))
        # cross attention
        # the masked self attention output is the query, the encoder output (memory) is the key and value
        attn2, _ = self.multihead_attn(Z, memory, memory, attn_mask=memory_mask, key_padding_mask=memory_key_padding_mask)
        Z = self.norm2(Z + self.dropout(attn2))
        ff = self.dropout(\
            self.linear2(\
                self.dropout(\
                    self.linear1(Z).relu()
                    )
                )
            )
        return self.norm3(Z + ff)

The memory argument in the decoder code above corresponds to the output of the encoder. For full flexibility, we let the user pass the appropriate masks to the forward() method. In general - you need to set the padding masks appropriately both for the memory and the target and set the tgt_mask to a causal mask
- Pytorch actually provides nn.TransformerEncoderLayer and nn.TransformerDecoderLayer out of the box, with the same arguments as above plus a few more - most importantly batch_first, which must be set to true if the batch dimension is first, plus the is_casusal argument for each attention mask, and an activation argument that defaults to relu. 
- Pytorch also provides 3 more transformer modules - 
  - nn.TransformerEncoder:
    - simply chains the desired number of encoder layers. constructor takes the encoder layer plus the desired number of layers num_layers and it clones the given encoder layer num_layers times. the constructor also takes an optional normalization layer, which (if provided) is applied to the final output
  - nn.TransformerDecoder:
    - same except it chains decoder layers instead of encoder layers
  - nn.Transformer:
    - creates an encoder and decoder (both with layer norm), and chains them

Building these out from scratch as well: Encoder, Decoder, and Transformer:

In [12]:
from copy import deepcopy

In [13]:
class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(encoder_layer) for _ in range(num_layers)])
        self.norm = norm
    
    def forward(self, src, mask=None, src_key_padding_mask=None):
        Z = src
        for layer in self.layers:
            Z = layer(Z, mask, src_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [14]:
class TransformerDecoder(nn.Module):
    def __init__(self, decoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(decoder_layer) for _ in range(num_layers)])
        self.norm = norm
    
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        Z = tgt 
        for layer in self.layers:
            Z = layer(Z, memory, tgt_mask, memory_mask, tgt_key_padding_mask, memory_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [15]:
class Transformer(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        norm1 = nn.LayerNorm(d_model)
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers, norm1)

        decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
        norm2 = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers, norm2)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None, src_key_padding_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        memory = self.encoder(src, src_mask, src_key_padding_mask)
        output = self.decoder(tgt, memory, tgt_mask, memory_mask, tgt_key_padding_mask, memory_key_padding_mask)
        return output

Notes:
- The computed scores are of shape (B, h, Lq, Lk) -> for each head in each batch, every (Lq, Lk) matrix answers the question of for each query position i, how much should I attend to key position j.
- The scores matrix is initially computed even with the padding token values - but pad tokens should never be attended to, by ANY query 

Building a full transformer from scratch -  complete. Lets now use a transformer model to translate english to spanish.

**Building an English-to-Spanish Transformer:**
- building an english to spanish transformer model -> we use the defined PositionalEmbedding module and Pytorch's nn.Transformer. The pytorch implementation works faster than the custom one

In [16]:
z = torch.tensor([1,1,0])

In [19]:
z = z.bool()
z

tensor([ True,  True, False])

In [20]:
~z

tensor([False, False,  True])

In [21]:
size = [200]*2
size

[200, 200]

In [23]:
fm = torch.full(size, True)

In [24]:
fm

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        ...,
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [25]:
cm = torch.triu(fm, diagonal=1)

In [26]:
cm

tensor([[False,  True,  True,  ...,  True,  True,  True],
        [False, False,  True,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ..., False,  True,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False, False]])

In [28]:
nn.Transformer.generate_square_subsequent_mask(5, dtype=bool)

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])

In [55]:
class NmtTransformer(nn.Module):
    def __init__(self, vocab_size, max_length, embed_dim=512, pad_id=0, num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.pos_embed = PositionalEmbedding(max_length, embed_dim, dropout)
        #self.transformer = nn.Transformer(embed_dim, num_heads, num_encoder_layers=num_layers, num_decoder_layers=num_layers, batch_first=True)
        self.transformer = Transformer(d_model=embed_dim, nhead=num_heads, num_encoder_layers=num_layers, num_decoder_layers=num_layers)
        self.output = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, pair):
        # source embeddings including positional encodings - for encoder
        src_embeds = self.pos_embed(self.embed(pair.src_token_ids))
        # target embeddings including positional encodings - for decoder
        tgt_embeds = self.pos_embed(self.embed(pair.tgt_token_ids))
        # compute the padding masks - know that the masks from the tokenizer are of shape: B,L and 1 where no padding and 0 where there's padding
        # we convert them to booleans and reverse this so its false where need not the mask and true where we need the mask
        src_pad_mask = ~pair.src_mask.bool()
        tgt_pad_mask = ~pair.tgt_mask.bool()
        # now need the full attention masks
        size = [pair.tgt_token_ids.size(1)]*2 # creating a list of: [seq_len, seq_len] -> where seq_len is the seq_len of current batch's target token ids
        full_mask = torch.full(size, True, device=tgt_pad_mask.device) # creating a matrix of only true values and of shape: [seq_len, seq_len]
        # now converting this to a causal mask for the decoder
        causal_mask = torch.triu(full_mask, diagonal=1) # converts it to the mask we are looking for - true means mask, keeps masking as required
        # now feed the inputs through the transformer
        out_decoder = self.transformer(src_embeds, tgt_embeds, src_key_padding_mask=src_pad_mask,\
                                       memory_key_padding_mask=src_pad_mask, tgt_mask=causal_mask,\
                                        tgt_key_padding_mask=tgt_pad_mask)
        return self.output(out_decoder).permute(0,2,1) # changing the output to B, vocab_size, seq_len

- the forward func takes an NmtPair as input (class is as the one defined and used with rnn encoders and decoders)
- the code uses the not operator (~) to invert both the source and target masks because they contain false for each padding token but nn.MultiHeadAttention expects true for tokens it should ignore
- Next we create a square matrix of shape [Lq, Lq] - full of true and we get all elements abive the main diagonal using the torch.triu function with the rest defaulting to false. this results in a causal mask that we can use as the tgt_mask for the transformer: it will use this mask for the masked self-attention layer. Alternatively, alternatively could call nn.Transformer.generate_square_subsequent_mask() to create the causal mask just passing it the sequence length (pair.tgt_token_ids.size(1)) and setting dtype to bool
- we then call the transformer, passing it the source and target embeddings, as well as the appropriate masks
- lastly, we pass the result through the output linear layer, and we permute the last 2 dimensions because nn.CrossEntropyLoss expects the class dimension to be dimension 1. 

Loading and processing the data:

In [56]:
from datasets import load_dataset
from torch.utils.data import random_split, DataLoader
import tokenizers

In [57]:
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa",
    split=["validation", "test"])
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

In [58]:
def train_eng_spa():  # a generator function to iterate over all training text
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

max_length = 500
vocab_size = 10_000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer.normalizer = tokenizers.normalizers.Lowercase()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)

In [59]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [60]:
from collections import namedtuple

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))

In [61]:
def nmt_collate_fn(batch):
    src_texts = [pair['source_text'] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids, src_mask,
                     tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

batch_size = 32
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size,
                             collate_fn=nmt_collate_fn)

In [62]:
from utils.early_stopping import EarlyStopping

Training the model...

In [63]:
torch.manual_seed(42)

nmt_tr_model = NmtTransformer(vocab_size,max_length, embed_dim=128, pad_id=0,
                              num_heads=4, num_layers=2, dropout=0.1).to(device)

# if device == "mps":
#     nmt_tr_model.transformer = Transformer(d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2)

In [64]:
xentropy = nn.CrossEntropyLoss(ignore_index=0, reduction='sum')
optimizer = torch.optim.NAdam(nmt_tr_model.parameters())
early_stopper = EarlyStopping(patience=5, checkpoint_path='nmt_tx.pt', restore_best_weights=True, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',patience=2,factor=0.9)

In [65]:
n_epochs = 20

train_loss = [0]*n_epochs
val_loss = [0]*n_epochs

for epoch in range(n_epochs):
    nmt_tr_model.train()
    # iterate through the training data
    for inputs,labels in nmt_train_loader:
        inputs,labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = nmt_tr_model(inputs)
        loss =  xentropy(out, labels)
        loss.backward()
        optimizer.step()
        train_loss[epoch] += loss.item()
    train_loss[epoch] /= len(nmt_train_loader.dataset)

    # eval on the val set
    nmt_tr_model.eval()
    with torch.no_grad():
        for inputs, labels in nmt_valid_loader:
            inputs,labels = inputs.to(device),labels.to(device)
            out = nmt_tr_model(inputs)
            loss = xentropy(out, labels)
            val_loss[epoch] += loss.item()
        val_loss[epoch] /= len(nmt_valid_loader.dataset)

        scheduler.step(val_loss[epoch])
        print(f'Epoch: {epoch+1}| Train loss: {train_loss[epoch]:.4f}| Val loss: {val_loss[epoch]:.4f}')
        early_stopper(val_loss[epoch], nmt_tr_model, optimizer, epoch)
        if early_stopper.should_stop:
            print(f"Stopping at epoch: {epoch+1}")
            break

Epoch: 1| Train loss: 42.6158| Val loss: 36.2925
Metric improved to 36.2925. Checkpoint saved at epoch 0
Epoch: 2| Train loss: 35.2228| Val loss: 32.0614
Metric improved to 32.0614. Checkpoint saved at epoch 1
Epoch: 3| Train loss: 31.7258| Val loss: 28.6684
Metric improved to 28.6684. Checkpoint saved at epoch 2
Epoch: 4| Train loss: 28.9467| Val loss: 26.3998
Metric improved to 26.3998. Checkpoint saved at epoch 3
Epoch: 5| Train loss: 26.9663| Val loss: 24.6541
Metric improved to 24.6541. Checkpoint saved at epoch 4
Epoch: 6| Train loss: 25.4210| Val loss: 23.3989
Metric improved to 23.3989. Checkpoint saved at epoch 5
Epoch: 7| Train loss: 24.2925| Val loss: 22.3579
Metric improved to 22.3579. Checkpoint saved at epoch 6
Epoch: 8| Train loss: 23.3637| Val loss: 21.8397
Metric improved to 21.8397. Checkpoint saved at epoch 7
Epoch: 9| Train loss: 22.6439| Val loss: 21.2542
Metric improved to 21.2542. Checkpoint saved at epoch 8
Epoch: 10| Train loss: 22.0143| Val loss: 20.8096
Metri

In [66]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text,
                                    "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index]  # take the last token ID

        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [69]:
nmt_tr_model.eval()
translate(nmt_tr_model, "I like to play soccer with my good friends at the beach, especially on Saturdays, but I don't really mind other days as well")

' me gusta jugar al fútbol con mi buen amigo , pero no me gusta el otro día , sino que'

Trained for only 20 epochs but got really impressive performance. Would have done way better if had been left to train longer...